## Step 1: Problem Framing

Urban parking spaces are limited resources with fluctuating demand throughout the day.
Static pricing leads to either overcrowding or underutilization.

### Objective
Design a **real-time dynamic pricing system** for multiple urban parking lots that:
- Updates prices continuously based on demand signals
- Ensures smooth and bounded price changes
- Accounts for competition from nearby parking lots
- Provides explainable pricing decisions

### Real-Time Constraint
At any time `t`, the system:
- Only has access to current and past data
- Cannot see future occupancy or demand
- Must make pricing decisions incrementally

### Tools Used
- Pathway for real-time streaming and windowed computation
- Kafka (simulated) for event ingestion
- Bokeh & Panel for real-time visualization
- Docker for deployment


# Step 2: Environment Setup (Pathway + Streaming).

In [1]:
# Install Pathway and other necessary libraries
!pip install pathway pandas bokeh panel faker --quiet

# For Kafka simulation (optional)
!pip install confluent-kafka --quiet

# Imports
import pathway as pw
import pandas as pd
from bokeh.plotting import figure, show, output_notebook
import panel as pn
from faker import Faker
import random
import time

# Enable Bokeh in Colab
output_notebook()
pn.extension()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 777.6/777.6 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.6/244.6 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.6/148.6 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
import numpy as np # Data Analysis
import matplotlib.pyplot as plt # Data Visualization
from datetime import datetime, timedelta

In [3]:
import pathway as pw

# List all available attributes/methods in the module
dir(pw)


['AsyncTransformer',
 'BaseCustomAccumulator',
 'ColumnExpression',
 'ColumnReference',
 'DateTimeNaive',
 'DateTimeUtc',
 'Duration',
 'GroupedJoinResult',
 'GroupedTable',
 'JoinMode',
 'JoinResult',
 'Joinable',
 'Json',
 'LiveTable',
 'MonitoringLevel',
 'PersistenceMode',
 'Pointer',
 'PyObjectWrapper',
 'Schema',
 'SchemaProperties',
 'Table',
 'TableLike',
 'TableSlice',
 'Type',
 'UDF',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 '_engine_finder',
 'annotations',
 'apply',
 'apply_async',
 'apply_with_type',
 'assert_table_has_schema',
 'cast',
 'coalesce',
 'column_definition',
 'debug',
 'declare_type',
 'demo',
 'enable_interactive_mode',
 'engine',
 'fill_error',
 'global_error_log',
 'graphs',
 'groupby',
 'if_else',
 'indexing',
 'internals',
 'io',
 'iterate',
 'iterate_universe',
 'join',
 'join_inner',
 'join_left',
 'join_outer',
 'join_right',
 'left',
 'lo

In [5]:
import pathway as pw

parking_table = pw.io.csv.read(
    "./dataset.csv",
    schema=pw.schema_from_csv("./dataset.csv"),

)
pw.io.csv.write(
    parking_table,
    "./out_parking",

)



/usr/local/lib/python3.12/dist-packages/beartype/_util/hint/pep/utilpeptest.py:311: BeartypeDecorHintPep585DeprecationWarning: PEP 484 type hint typing.Iterable[pathway.internals.expression.ColumnReference] deprecated by PEP 585. This hint is scheduled for removal in the first Python version released after October 5th, 2025. To resolve this, import this hint from "beartype.typing" rather than "typing". For further commentary and alternatives, see also:
    https://beartype.readthedocs.io/en/latest/api_roar/#pep-585-deprecations
  warn(


In [6]:
ls out_parking


ls: cannot access 'out_parking': No such file or directory


# Explicit schema

In [7]:
class ParkingSchema(pw.Schema):
    ID: int
    SystemCodeNumber: str
    Capacity: int
    Latitude: float
    Longitude: float
    Occupancy: int
    VehicleType: str
    TrafficConditionNearby: str
    QueueLength: int
    IsSpecialDay: int
    LastUpdatedDate: str
    LastUpdatedTime: str


# STEP 2 : Load Dataset in Batch Mode using the Schema

Goal of this step:

✔ Connect your real dataset to Pathway
✔ Validate it against the schema
✔ Prepare it for feature engineering

Important:

Batch mode lets us finish execution

Streaming would block everything (we’ll use it later)

In [8]:
parking_table = pw.io.csv.read(
    "./dataset.csv",
    schema=ParkingSchema
)


# STEP 3: Feature Engineering – Core Signal for Pricing

Dynamic pricing depends on demand pressure.

From your dataset:

Occupancy → how many vehicles parked

Capacity → max slots

The single most important feature is:

Occupancy Ratio = Occupancy / Capacity

Everything later (all 3 models) depends on this.





In [9]:
parking_table = parking_table.select(
    *parking_table,
    occupancy_ratio = parking_table.Occupancy / parking_table.Capacity
)


# STEP 4 : Model 1 – Baseline Linear Pricing

This is your first pricing model.

Simple, rule-based

Uses the occupancy_ratio we just engineered

Purpose: establish a baseline before dynamic or competitive pricing

🧠 Why Step 4 exists

Baseline gives a reference price

Helps check sanity of future models

Explains logic to stakeholders

In [10]:
parking_table = parking_table.select(
    *parking_table,
    linear_price = 10 * (1 + parking_table.occupancy_ratio)
)


In [11]:
print(parking_table.schema)  # Shows column names + types


id          | ID  | SystemCodeNumber | Capacity | Latitude | Longitude | Occupancy | VehicleType | TrafficConditionNearby | QueueLength | IsSpecialDay | LastUpdatedDate | LastUpdatedTime | occupancy_ratio | linear_price
ANY_POINTER | INT | STR              | INT      | FLOAT    | FLOAT     | INT       | STR         | STR                    | INT         | INT          | STR             | STR             | FLOAT           | FLOAT       


In [12]:
# This will capture all columns up to Step 4
pw.io.csv.write(
    parking_table,
    "./temp_check_step4"
)

# STEP 5 : Model 2 – Dynamic Pricing (Demand-Based)

This model adjusts price based on occupancy pressure — classic surge pricing.

🧠 Why Step 5 exists

Baseline ignores high demand

Dynamic model reacts only to current occupancy ratio

Simple real-time rule:

In [13]:
parking_table = parking_table.select(
    *parking_table,
    dynamic_price = pw.if_else(
        parking_table.occupancy_ratio > 0.8,
        10 * 1.5,
        10 * 1.1
    )
)


In [14]:
print(parking_table.schema)


id          | ID  | SystemCodeNumber | Capacity | Latitude | Longitude | Occupancy | VehicleType | TrafficConditionNearby | QueueLength | IsSpecialDay | LastUpdatedDate | LastUpdatedTime | occupancy_ratio | linear_price | dynamic_price
ANY_POINTER | INT | STR              | INT      | FLOAT    | FLOAT     | INT       | STR         | STR                    | INT         | INT          | STR             | STR             | FLOAT           | FLOAT        | FLOAT        


In [15]:
# Vehicle & traffic mapping UDFs
vehicle_map = {"car": 1.0, "bike": 0.8}
traffic_map = {"low": 1.0, "medium": 1.5, "high": 2.0}

In [16]:
def map_vehicle(v):
    try:
        return float(vehicle_map.get(v, 1.0))
    except:
        return 1.0


In [17]:
def map_traffic(t):
    try:
        return float(traffic_map.get(t, 1.0))
    except:
        return 1.0

In [18]:
# Compute numeric weights only (no duplicates)
parking_table = parking_table.select(
    *parking_table,
    VehicleTypeWeightNum = pw.udf(lambda v: float(vehicle_map.get(v, 1.0)))(parking_table.VehicleType),
    TrafficWeight = pw.udf(lambda t: float(traffic_map.get(t, 1.0)))(parking_table.TrafficConditionNearby),
    QueueLengthFloat = pw.udf(lambda q: float(q))(parking_table.QueueLength),
    IsSpecialDayFloat = pw.udf(lambda s: float(s))(parking_table.IsSpecialDay)
)

In [19]:
# Parameters
alpha, beta, gamma, delta, epsilon = 0.6, 0.2, 0.1, 0.1, 0.1
lambda_price = 1.0
base_price = 10.0

In [20]:
# Compute raw demand
parking_table = parking_table.select(
    *parking_table,
    raw_demand = pw.udf(
        lambda occ, qlen, tweight, special, vweight: (
            alpha*occ + beta*qlen - gamma*tweight + delta*special + epsilon*vweight
        )
    )(
        parking_table.occupancy_ratio,  # reuse existing
        parking_table.QueueLengthFloat,
        parking_table.TrafficWeight,
        parking_table.IsSpecialDayFloat,
        parking_table.VehicleTypeWeightNum
    )
)

In [21]:
# Normalize demand 0-1
parking_table = parking_table.select(
    *parking_table,
    normalized_demand = pw.udf(lambda d: max(0, min(1, d)))(parking_table.raw_demand)
)

In [22]:
# Dynamic price (bounded 0.5x-2x base) — use a **new column name** to avoid duplicates
parking_table = parking_table.select(
    *parking_table,
    dynamic_price_step5 = pw.udf(lambda d: max(0.5*base_price, min(2*base_price, base_price*(1+d*lambda_price))))(
        parking_table.normalized_demand
    )
)

In [23]:
# Write output (optional)
pw.io.csv.write(parking_table, "./pricing_output_step5")

# Step 6: Competitive Pricing

Logic

Assume competitor prices are provided per lot (competitor_price column).

If not available, we can simulate random nearby prices.

Adjust price based on occupancy vs competitor:

If your lot is nearly full and competitors are cheaper → reduce price (attract customers)

If competitors are expensive → you can increase price (profit)

Keep a new column to avoid duplicates: competitive_price_step6

In [24]:

import random

# Simulate competitor price if not in dataset
# For demo, random price around base price
def simulate_competitor_price(base):
    return round(base * random.uniform(0.8, 1.2), 2)

parking_table = parking_table.select(
    *parking_table,
    competitor_price = pw.udf(simulate_competitor_price)(parking_table.linear_price)
)

# Competitive pricing logic
def competitive_logic(dynamic_price, occupancy_ratio, competitor_price):
    if occupancy_ratio > 0.9 and competitor_price < dynamic_price:
        # Lot is nearly full, competitors cheaper → reduce price slightly
        return max(0.5*dynamic_price, dynamic_price*0.9)
    elif competitor_price > dynamic_price:
        # Competitors more expensive → increase price
        return min(2*dynamic_price, dynamic_price*1.1)
    else:
        # Otherwise, keep dynamic price
        return dynamic_price

# Create new competitive price column
parking_table = parking_table.select(
    *parking_table,
    competitive_price_step6 = pw.udf(competitive_logic)(
        parking_table.dynamic_price_step5,   # previous dynamic price
        parking_table.occupancy_ratio,
        parking_table.competitor_price
    )
)

# Optional: write intermediate output (can skip pw.run() for now)
pw.io.csv.write(parking_table, "./pricing_output_step6")


# Step 7: Bokeh Visualization

In [25]:
pw.io.csv.write(
    parking_table,
    "./pricing_output_step7"
)

# Run Pathway to mate

In [ ]:
import pathway as pw

pw.run()

In [28]:
import pandas as pd
df = pd.read_csv("./pricing_output_step7")

In [29]:
df.head()

,ID,SystemCodeNumber,Capacity,Latitude,Longitude,Occupancy,VehicleType,TrafficConditionNearby,QueueLength,IsSpecialDay,...,TrafficWeight,QueueLengthFloat,IsSpecialDayFloat,raw_demand,normalized_demand,dynamic_price_step5,competitor_price,competitive_price_step6,time,diff
0,2989,BHMEURBRD01,470,26.149020,91.739503,403,car,low,4,0,...,1.0,4.0,0.0,1.314468,1.000000,20.000000,16.61,20.000000,1773284701718,1
1,6212,BHMNCPHST01,1200,26.140014,91.731000,892,bike,high,9,0,...,2.0,9.0,0.0,2.126000,1.000000,20.000000,18.34,20.000000,1773284701718,1
2,13725,Others-CCCPS202,2937,26.147491,91.727997,1370,car,high,9,0,...,2.0,9.0,0.0,1.979877,1.000000,20.000000,13.31,20.000000,1773284701718,1
3,7762,BHMNCPNST01,485,26.140048,91.730972,270,car,low,2,0,...,1.0,2.0,0.0,0.734021,0.734021,17.340206,15.96,17.340206,1773284701718,1
4,8089,Broad Street,690,26.137958,91.740994,96,bike,low,2,1,...,1.0,2.0,1.0,0.563478,0.563478,15.634783,12.16,15.634783,1773284701718,1


In [30]:
import pandas as pd
from bokeh.plotting import figure, show


p = figure(title="Pricing Comparison", x_axis_label="Row Index", y_axis_label="Price")
p.line(df.index, df["linear_price"], color="blue", legend_label="Linear")
p.line(df.index, df["dynamic_price_step5"], color="green", legend_label="Dynamic")
p.line(df.index, df["competitive_price_step6"], color="red", legend_label="Competitive")

show(p)

In [31]:
# Read the same dataset in streaming mode
parking_table_stream = pw.io.csv.read(
    "./dataset.csv",
    schema=ParkingSchema,
    mode="streaming"
)

# Now your table updates automatically as new rows arrive
# (pw.run() in this mode will keep running continuously)


# Step 9: Simulated Real-Time Streaming Table

In [32]:
import pathway as pw
import pandas as pd
import time

# 1️⃣ Split your dataset into small chunks to simulate streaming
df = pd.read_csv("./dataset.csv")

batch_size = 500  # number of rows per "stream event"
batches = [df.iloc[i:i+batch_size] for i in range(0, len(df), batch_size)]

# 2️⃣ Define Pathway Schema exactly as before
class ParkingSchema(pw.Schema):
    ID: int
    SystemCodeNumber: str
    Capacity: int
    Latitude: float
    Longitude: float
    Occupancy: int
    VehicleType: str
    TrafficConditionNearby: str
    QueueLength: int
    IsSpecialDay: int
    LastUpdatedDate: str
    LastUpdatedTime: str
    base_price: float     # default base price
    competitor_price: float  # default competitor price

# 3️⃣ Prepare empty Pathway table
stream_table = None


# Step 10: Convert to Real-Time Streaming

Now we take your batch pipeline and make it real-time / streaming, so Pathway updates prices automatically as new events arrive.

We do not rerun previous steps manually — we reuse all the logic from Step 4–9, just switch the table source to streaming mode.

In [33]:
import pathway as pw

# Reuse your existing schema
# Replace `ParkingSchema` with your schema class if you have one
parking_table_stream = pw.io.csv.read(
    "./dataset.csv",
    schema=ParkingSchema,  # exact schema from Step 2
    mode="streaming"       # enables real-time updates
)


In [34]:
# Occupancy ratio
parking_table_stream = parking_table_stream.select(
    *parking_table_stream,
    occupancy_ratio = parking_table_stream.Occupancy / parking_table_stream.Capacity
)

# Model 1: Linear pricing
parking_table_stream = parking_table_stream.select(
    *parking_table_stream,
    linear_price = parking_table_stream.base_price * (1 + parking_table_stream.occupancy_ratio)
)

# Model 2: Dynamic pricing
parking_table_stream = parking_table_stream.select(
    *parking_table_stream,
    dynamic_price = (
        parking_table_stream.base_price *
        (1 + 0.1 * parking_table_stream.occupancy_ratio)  # replace with your demand function if available
    )
)

# Model 3: Competitive pricing (optional)
parking_table_stream = parking_table_stream.select(
    *parking_table_stream,
    competitive_price = pw.if_else(
        parking_table_stream.base_price > parking_table_stream.competitor_price,
        parking_table_stream.competitor_price - 1,
        parking_table_stream.base_price + 1
    )
)


In [35]:
# Persist streaming output
pw.io.csv.write(
    parking_table_stream,
    "./pricing_output_streaming"
)


In [36]:
import pathway as pw

parking_table_stream = pw.io.csv.read(
    "./dataset.csv",
    schema=ParkingSchema,
    mode="streaming"
)
